# Live Music Diffusion — Inference

Minimal example: load a model, then run streaming block-AR generation.


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["ENABLE_TORCH_COMPILE"] = "1"

import json
import torch
import torchaudio
from einops import rearrange
import IPython.display as ipd

from stable_audio_tools.models.factory import create_model_from_config
from stable_audio_tools.models.utils import copy_state_dict, load_ckpt_state_dict
from stable_audio_tools.inference.generation import generate_diffusion_cond_blockar

device = "cuda" if torch.cuda.is_available() else "cpu"

# --- Point these at your config + checkpoint ---
model_config_path = "stable_audio_tools/configs/model_configs/txt2audio/saos_arc_forcing_block_causal.json"
ckpt_path = "/path/to/model.ckpt"

with open(model_config_path) as f:
    model_config = json.load(f)

model = create_model_from_config(model_config)
state = load_ckpt_state_dict(ckpt_path)

# Split fused qkv projections for KV-cache streaming. If the checkpoint was saved
# already-split, split first so the keys line up; otherwise load then split.
already_split = any("self_attn.to_q.weight" in k for k in state.keys())
def split_qkv():
    for m in model.model.model.transformer.modules():
        if hasattr(m, "_split_qkv_projections_for_cache"):
            m._split_qkv_projections_for_cache()
if already_split:
    split_qkv()
copy_state_dict(model, state)
if not already_split:
    split_qkv()

model = model.to(device).eval().requires_grad_(False).to(torch.float16)

sample_rate = model_config["sample_rate"]
sample_size = model_config["sample_size"]
attn_pattern = model_config["training"]["inpainting"]["mask_kwargs"].get(
    "context_router_attention_pattern", "enc-dec")
print("Loaded model | attention pattern:", attn_pattern)


In [ ]:
conditioning = [{
    "prompt": "chill lo-fi beats, laid-back drums, jazzy chords, 90 BPM, lowpass filter",
    "seconds_start": 0,
    "seconds_total": 12,
}]

block_size = 96256          # audio samples generated per block (47 latents)
n_blocks = 20               # total blocks to generate

with torch.inference_mode():
    with torch.cuda.amp.autocast(enabled=True, dtype=torch.float16):
        output = generate_diffusion_cond_blockar(
            model,
            steps=8,
            cfg_scale=0.95,
            conditioning=conditioning,
            sample_size=sample_size,
            sampler_type="pingpong",
            sigma_min=0,
            sigma_max=1,
            device=device,
            ar_style="outpaint",
            block_size=block_size,
            generation_length=block_size * n_blocks,
            seed=89158122249099,
            context_router=True,
            context_router_attention_pattern=attn_pattern,
            postpend=True,
            use_kv_cache=True,
            prefill=True,
            plus_plus=True,
        )

output = rearrange(output, "b d n -> d (b n)")
output = output.to(torch.float32).div(output.abs().max().clamp(min=1e-8)).clamp(-1, 1).cpu()
torchaudio.save("output.wav", output, sample_rate)
ipd.Audio(output.numpy(), rate=sample_rate)
